# 03 — Modelling and stress tests

The classifier, and the analyses that try to break it. Read the ablation
table before the headline AUC — if the technical-only row is not at
chance, the headline AUC does not mean what it appears to mean.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np, pandas as pd
from cfdna_insight.features import load_features

FEATURES = '../results/features'   # or '../results/smoke/features'
blocks = load_features(FEATURES)
samples = pd.read_csv(os.path.join(FEATURES, 'samples_used.tsv'), sep='\t')
order = [s for s in samples['sample_id'] if s in blocks['size_summary'].index]
samples = samples.set_index('sample_id').loc[order].reset_index()
y = (samples['label'] == 'cancer').astype(int).to_numpy()
{k: v.shape for k, v in blocks.items()}

In [ ]:
from cfdna_insight import modeling, plots
from cfdna_insight.cnv import HealthyReference, arm_summary

if 'coverage' in blocks:
    ref = HealthyReference.fit(blocks['coverage'].reindex(order)[y == 0])
    arms = blocks['window_arms'].reindex(order).iloc[0]
    blocks['cnv'] = pd.DataFrame({sid: arm_summary(
        ref.zscore(blocks['coverage'].loc[sid]), arms) for sid in order}).T
    blocks['cnv'].shape

## Ablation, negative control first

The row to read first is `TECHNICAL ONLY`. It should be at 0.5.

In [ ]:
abl = modeling.ablation(blocks, y, order, n_repeats=10)
abl[['label', 'n_features', 'auc_mean', 'auc_lo', 'auc_hi']]

In [ ]:
plots.plot_ablation(abl, '/tmp/abl.png')
from IPython.display import Image; Image('/tmp/abl.png')

## Full model

In [ ]:
bio = [b for b in ('size_summary','size_hist','delfi','cnv','motif') if b in blocks]
X = modeling.assemble_matrix(blocks, bio, order)
res = modeling.nested_cv(X, y, n_repeats=10, label='full')
lo, hi = res.auc_ci
print(f'AUC {res.auc_mean:.3f}  [{lo:.3f}, {hi:.3f}] across CV repeats')
print(f'({X.shape[1]} features, n = {len(y)})')

## Permutation null

The full nested CV is re-run under shuffled labels. This is the reference
distribution the AUC above should be judged against.

In [ ]:
null, p = modeling.permutation_null(X, y, n_perm=200, n_repeats=2)
print(f'null: mean {null.mean():.3f}, sd {null.std():.3f}, 95th pct {np.percentile(null,95):.3f}')
print(f'observed {res.auc_mean:.3f}, permutation p = {p:.4g}')
plots.plot_permutation_null(null, res.auc_mean, p, '/tmp/perm.png')
from IPython.display import Image; Image('/tmp/perm.png')

## Learning curve

Still climbing at the full sample size means the estimate has not
converged and should not be quoted as if it had.

In [ ]:
modeling.learning_curve(X, y)

## Cross-study generalisation

The number that would actually matter for a screening assay.

In [ ]:
groups = samples['study'].to_numpy()
if len(np.unique(groups)) >= 2:
    cs = modeling.cross_study_cv(X, y, groups)
    print(f'held-out-study AUC {cs.auc_mean:.3f}')
    print(cs.extra['per_study_auc'])
else:
    print('single study — cross-study generalisation is UNTESTED.')
    print('The AUC above is within-cohort only and must be reported that way.')

## What the model is using

In [ ]:
final = modeling.fit_final(X, y)
modeling.coefficient_report(final, list(X.columns), top_n=25)

---
### Writing this up

Report, in this order: the ablation table including the negative control,
the permutation p, the cross-study AUC (or the fact that it is untested),
the learning curve, and only then the headline AUC. A null or weak result
that has been properly stress-tested is a stronger portfolio piece than a
strong result that has not been.